<a href="https://colab.research.google.com/github/merence-DA/Ecommerce-Analytics-SQL/blob/main/A_B_Testing_Using_Statistical_Methods_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Setting Up the Environment and Authorizing in Google Cloud**

In [13]:
!pip install google-cloud-bigquery pandas-gbq

from google.colab import auth
from google.cloud import bigquery
import pandas as pd

auth.authenticate_user()

PROJECT_ID = 'data-analytics-mate'
client = bigquery.Client(project=PROJECT_ID)

print("Authentication was successful.")

Authentication was successful.


**Step 2: Exporting and Consolidating Data from BigQuery**

In [4]:
# SQL-request

sql_query = """
with session_info as (
  select
          s.date,
          s.ga_session_id,
          sp.country,
          sp.device,
          sp.continent,
          sp.channel,
          ab.test,
          ab.test_group
  from `DA.ab_test` as ab
  join `DA.session` as s
  on ab.ga_session_id = s.ga_session_id
  join `DA.session_params` as sp
  on sp.ga_session_id = ab.ga_session_id
),
session_with_orders as (
select
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count (distinct o.ga_session_id) as session_with_orders
from `data-analytics-mate.DA.order` as o
join session_info
on o.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
events as (
select
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name,
        count (ep.ga_session_id) as event_cnt
from `data-analytics-mate.DA.event_params` as ep
join session_info
on ep.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name
),
sessions as (
select
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count (distinct session_info.ga_session_id) as session_cnt
from session_info
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),
account as (
select
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        count (distinct acs.ga_session_id) as new_account_cnt
from `data-analytics-mate.DA.account_session` as acs
join session_info
on acs.ga_session_id = session_info.ga_session_id
group by
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
)
select
        session_with_orders.date,
        session_with_orders.country,
        session_with_orders.device,
        session_with_orders.continent,
        session_with_orders.channel,
        session_with_orders.test,
        session_with_orders.test_group,
        'session_with_orders' as event_name,
        session_with_orders.session_with_orders as value
from session_with_orders
union all
select
        events.date,
        events.country,
        events.device,
        events.continent,
        events.channel,
        events.test,
        events.test_group,
        event_name,
        event_cnt as value
from events
union all
select
        sessions.date,
        sessions.country,
        sessions.device,
        sessions.continent,
        sessions.channel,
        sessions.test,
        sessions.test_group,
        'session' as event_name,
        session_cnt as value
from sessions
union all
select
        account.date,
        account.country,
        account.device,
        account.continent,
        account.channel,
        account.test,
        account.test_group,
        'new account' as event_name,
        new_account_cnt as value
from account

"""

df_analytics = client.query(sql_query).to_dataframe(
    create_bqstorage_client=False
)

print(f"Dataset size: {df_analytics.shape[0]} rows, {df_analytics.shape[1]} columns.")

df_analytics.head()

Dataset size: 800996 rows, 9 columns.


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Lithuania,mobile,Europe,Organic Search,2,2,new account,1
1,2020-11-01,El Salvador,desktop,Americas,Social Search,2,1,new account,1
2,2020-11-01,Slovakia,mobile,Europe,Paid Search,2,2,new account,1
3,2020-11-01,Lithuania,desktop,Europe,Paid Search,2,2,new account,1
4,2020-11-02,North Macedonia,desktop,Europe,Direct,2,1,new account,1


**Step 3: Calculating the overall significance of the 4 metrics for the test**

In [5]:
df_grouped = df_analytics.groupby(['test', 'test_group', 'event_name'])['value'].sum().reset_index()
df_pivot = df_grouped.pivot_table(index=['test', 'test_group'],
                                  columns='event_name',
                                  values='value').reset_index()
df_pivot.head()

event_name,test,test_group,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,new account,page_view,...,select_item,select_promotion,session,session_start,session_with_orders,user_engagement,view_item,view_item_list,view_promotion,view_search_results
0,1,1,1988.0,3034.0,1395.0,3784.0,368.0,30596.0,3823.0,191543.0,...,543.0,1275.0,45362.0,45905.0,4514.0,171788.0,62335.0,27.0,29188.0,3678.0
1,1,2,2229.0,3221.0,1366.0,4021.0,353.0,30512.0,3681.0,198050.0,...,530.0,1323.0,45193.0,45649.0,4526.0,179081.0,65337.0,24.0,29117.0,3882.0
2,2,1,2344.0,3480.0,2811.0,4262.0,337.0,34511.0,4165.0,220275.0,...,905.0,1477.0,50637.0,51219.0,5102.0,198266.0,72717.0,24.0,32367.0,4282.0
3,2,2,2409.0,3510.0,3061.0,4313.0,413.0,34171.0,4184.0,212320.0,...,946.0,1406.0,50244.0,50808.0,5003.0,189931.0,68700.0,29.0,31680.0,4198.0
4,3,1,3623.0,5298.0,17674.0,9532.0,280.0,50438.0,5856.0,286351.0,...,8735.0,2020.0,70047.0,71312.0,6951.0,249921.0,93931.0,9.0,41169.0,5764.0


In [7]:
metrics_to_calculate = [
    {'name': 'add_payment', 'numerator': 'add_payment_info', 'denominator': 'session'},
    {'name': 'add_shipping', 'numerator': 'add_shipping_info', 'denominator': 'session'},
    {'name': 'begin_checkout', 'numerator': 'begin_checkout', 'denominator': 'session'},
    {'name': 'new_accounts', 'numerator': 'new account', 'denominator': 'session'}
]

tests = df_pivot['test'].unique()

In [8]:
from statsmodels.stats.proportion import proportions_ztest

results = []

for test_id in tests:
    test_data = df_pivot[df_pivot['test'] == test_id]
    if len(test_data) < 2:
        continue

    for metric in metrics_to_calculate:
        ctrl_row = test_data[test_data['test_group'] == 1]
        test_row = test_data[test_data['test_group'] == 2]

        n_ctrl = ctrl_row[metric['numerator']].values[0]
        d_ctrl = ctrl_row[metric['denominator']].values[0]
        n_test = test_row[metric['numerator']].values[0]
        d_test = test_row[metric['denominator']].values[0]

        # Calculating the Z-test
        z_stat, p_val = proportions_ztest([n_test, n_ctrl], [d_test, d_ctrl])

        # Conversions
        cr_ctrl = n_ctrl / d_ctrl
        cr_test = n_test / d_test

        results.append({
            'test_number': test_id,
            'metric': metric['name'],
            # Add event names
            'numerator_event': metric['numerator'],
            'denominator_event': metric['denominator'],
            # Test group data
            'numerator_test': n_test,
            'denominator_test': d_test,
            'conversion_rate_test': cr_test,
            # Control group data
            'numerator_control': n_ctrl,
            'denominator_control': d_ctrl,
            'conversion_rate_control': cr_ctrl,
            # Statistic
            'metric_change': (cr_test - cr_ctrl) / cr_ctrl * 100,
            'z_stat': z_stat,
            'p_value': p_val,
            'significant': p_val < 0.05
        })

df_final = pd.DataFrame(results)
df_final

,test_number,metric,numerator_event,denominator_event,numerator_test,denominator_test,conversion_rate_test,numerator_control,denominator_control,conversion_rate_control,metric_change,z_stat,p_value,significant
0,1,add_payment,add_payment_info,session,2229.0,45193.0,0.049322,1988.0,45362.0,0.043825,12.542021,3.924884,0.000087,True
1,1,add_shipping,add_shipping_info,session,3221.0,45193.0,0.071272,3034.0,45362.0,0.066884,6.560481,2.603571,0.009226,True
2,1,begin_checkout,begin_checkout,session,4021.0,45193.0,0.088974,3784.0,45362.0,0.083418,6.660587,2.978783,0.002894,True
3,1,new_accounts,new account,session,3681.0,45193.0,0.081451,3823.0,45362.0,0.084278,-3.354299,-1.542883,0.122859,False
4,2,add_payment,add_payment_info,session,2409.0,50244.0,0.047946,2344.0,50637.0,0.046290,3.576911,1.240994,0.214608,False
5,2,add_shipping,add_shipping_info,session,3510.0,50244.0,0.069859,3480.0,50637.0,0.068724,1.650995,0.709557,0.477979,False
6,2,begin_checkout,begin_checkout,session,4313.0,50244.0,0.085841,4262.0,50637.0,0.084168,1.988164,0.952898,0.340642,False
7,2,new_accounts,new account,session,4184.0,50244.0,0.083274,4165.0,50637.0,0.082252,1.241934,0.588793,0.556000,False
8,3,add_payment,add_payment_info,session,3697.0,70439.0,0.052485,3623.0,70047.0,0.051722,1.474630,0.643172,0.520112,False
9,3,add_shipping,add_shipping_info,session,5188.0,70439.0,0.073652,5298.0,70047.0,0.075635,-2.621211,-1.413727,0.157442,False


In [9]:
from google.colab import drive
drive.mount('/content/drive')
df_final.to_csv('/content/drive/MyDrive/ab_test_results.csv', index=False)

Mounted at /content/drive


**Step 4: Calculating the significance of the 4 metrics by country**

In [10]:
df_grouped_country = df_analytics.groupby(['test', 'test_group', 'country', 'event_name'])['value'].sum().reset_index()

df_pivot_country = df_grouped_country.pivot_table(
    index=['test', 'test_group', 'country'],
    columns='event_name',
    values='value'
).reset_index().fillna(0)
df_pivot_country.head()

event_name,test,test_group,country,add_payment_info,add_shipping_info,add_to_cart,begin_checkout,click,first_visit,new account,...,select_item,select_promotion,session,session_start,session_with_orders,user_engagement,view_item,view_item_list,view_promotion,view_search_results
0,1,1,(not set),16.0,23.0,10.0,26.0,5.0,245.0,29.0,...,11.0,9.0,369.0,372.0,38.0,1286.0,397.0,0.0,254.0,22.0
1,1,1,Albania,1.0,2.0,4.0,3.0,0.0,4.0,0.0,...,0.0,0.0,9.0,9.0,1.0,27.0,26.0,0.0,5.0,1.0
2,1,1,Algeria,2.0,1.0,0.0,1.0,0.0,20.0,1.0,...,0.0,0.0,29.0,30.0,5.0,99.0,25.0,0.0,20.0,1.0
3,1,1,Argentina,6.0,5.0,9.0,5.0,1.0,93.0,6.0,...,1.0,2.0,122.0,129.0,15.0,444.0,182.0,0.0,77.0,9.0
4,1,1,Armenia,0.0,1.0,0.0,1.0,0.0,11.0,1.0,...,0.0,0.0,8.0,11.0,1.0,121.0,60.0,0.0,9.0,2.0


In [12]:
from statsmodels.stats.proportion import proportions_ztest
import warnings

warnings.filterwarnings('ignore')

country_results = []
unique_countries = df_pivot_country['country'].unique()

for country in unique_countries:
    country_data = df_pivot_country[df_pivot_country['country'] == country]

    for test_id in country_data['test'].unique():
        test_group_data = country_data[country_data['test'] == test_id]

        if len(test_group_data) < 2:
            continue

        for metric in metrics_to_calculate:
            ctrl = test_group_data[test_group_data['test_group'] == 1]
            test = test_group_data[test_group_data['test_group'] == 2]

            n_ctrl = ctrl[metric['numerator']].values[0]
            d_ctrl = ctrl[metric['denominator']].values[0]
            n_test = test[metric['numerator']].values[0]
            d_test = test[metric['denominator']].values[0]

            # CRITERION: We perform the calculation only if there are at least 10 sessions in both groups
            if d_ctrl < 10 or d_test < 10:
                continue

            try:
                # Calculating the Z-test
                z_stat, p_val = proportions_ztest([n_test, n_ctrl], [d_test, d_ctrl])

                country_results.append({
                    'test_number': test_id,
                    'country': country,
                    'metric': metric['name'],
                    'numerator_event': metric['numerator'],
                    'conv_test': n_test / d_test,
                    'conv_ctrl': n_ctrl / d_ctrl,
                    'p_value': p_val,
                    'significant': p_val < 0.05
                })
            except:
                continue

df_final_countries = pd.DataFrame(country_results)

if not df_final_countries.empty:
    display(df_final_countries.head(40))
    print(f"\nThe results for have been successfully calculated {df_final_countries['country'].nunique()} countries.")
else:
    print("There isn't enough data to generate the table.")

,test_number,country,metric,numerator_event,conv_test,conv_ctrl,p_value,significant
0,1,(not set),add_payment,add_payment_info,0.050938,0.043360,0.626381,False
1,1,(not set),add_shipping,add_shipping_info,0.069705,0.062331,0.685902,False
2,1,(not set),begin_checkout,begin_checkout,0.096515,0.070461,0.199733,False
3,1,(not set),new_accounts,new account,0.075067,0.078591,0.856983,False
4,2,(not set),add_payment,add_payment_info,0.042755,0.059896,0.269240,False
5,2,(not set),add_shipping,add_shipping_info,0.061758,0.072917,0.527298,False
6,2,(not set),begin_checkout,begin_checkout,0.083135,0.083333,0.991898,False
7,2,(not set),new_accounts,new account,0.087886,0.070312,0.357268,False
8,3,(not set),add_payment,add_payment_info,0.069492,0.034483,0.007076,True
9,3,(not set),add_shipping,add_shipping_info,0.071186,0.063793,0.614329,False



The results for have been successfully calculated 108 countries.


**[Results of the calculation of the overall significance of the metrics for the test](https://docs.google.com/spreadsheets/d/1fEUj7kVd1A-HP2Fvx_61hFyd6TP_CJqK3UX7WNbvvxI/edit?usp=sharing)**

**[Dashboard in Tableau](https://public.tableau.com/views/PortfolioProject2_17732417256260/PortfolioProject22?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)**